# TinyZero-GRPO: Pure PyTorch GRPO on Countdown (Tesla T4 16 GB)

1-click turnkey pipeline: environment setup -> fast unit tests -> GRPO RLVR training + KL ablation (`Qwen/Qwen2.5-1.5B-Instruct`) -> plots, Markdown summary table, before/after reasoning traces, and artifact download.

In [ ]:
!nvidia-smi
import os
if not os.path.isdir("src"):
    if not os.path.isdir("tinyzero-grpo"):
        !git clone https://github.com/Medmad710/tinyzero-grpo.git
    %cd tinyzero-grpo
!pip install -q -r requirements.txt

In [ ]:
!PYTHONPATH=. pytest tests/ -v

In [ ]:
!PYTHONPATH=. python experiments/run_grpo.py

In [ ]:
!PYTHONPATH=. python experiments/plot_results.py

import json
import zipfile
from pathlib import Path
from IPython.display import Image, display

for img_path in ["assets/grpo_learning_curve.png", "assets/grpo_ablation.png"]:
    if Path(img_path).exists():
        print(f"\n=== {img_path} ===")
        display(Image(filename=img_path))

results = json.loads(Path("experiments/results.json").read_text(encoding="utf-8"))
main_run = results["main_run"]
eval_hist = main_run["eval_history"]
eval_0 = eval_hist[0]
eval_last = eval_hist[-1]
sys_info = main_run["system"]

ablation = results.get("ablation")
if ablation:
    wo_eval = ablation["without_kl"]["eval_history"][-1]
    wo_kl_last = ablation["without_kl"]["train_history"][-1]["kl"]
    w_kl_last = ablation["with_kl"]["train_history"][-1]["kl"]
else:
    wo_eval = None

print("\n### Copy-Pasteable Markdown Key Results Table\n")
print("| Run / Checkpoint | Held-Out pass@1 | Format Rate | Mean Reward | Mean Len (tokens) | Peak VRAM | Runtime |")
print("|---|---:|---:|---:|---:|---:|---:|")
print(
    f"| Baseline (Step 0, {sys_info['gpu_name']}) | {eval_0['pass_at_1']*100:.1f}% | "
    f"{eval_0['format_rate']*100:.1f}% | {eval_0['mean_reward']:.3f} | "
    f"{eval_0['mean_completion_len']:.1f} | - | - |"
)
print(
    f"| GRPO (`beta_KL=0.04`, Step {eval_last['step']}) | {eval_last['pass_at_1']*100:.1f}% | "
    f"{eval_last['format_rate']*100:.1f}% | {eval_last['mean_reward']:.3f} | "
    f"{eval_last['mean_completion_len']:.1f} | {sys_info['peak_vram_gb']:.2f} GB | "
    f"{sys_info['runtime_sec']/60:.1f} min |"
)
if wo_eval:
    wo_sys = ablation["without_kl"]["system"]
    print(
        f"| Ablation (`beta_KL=0.0`, Step {wo_eval['step']}) | {wo_eval['pass_at_1']*100:.1f}% | "
        f"{wo_eval['format_rate']*100:.1f}% | {wo_eval['mean_reward']:.3f} | "
        f"{wo_eval['mean_completion_len']:.1f} | {wo_sys['peak_vram_gb']:.2f} GB | "
        f"{wo_sys['runtime_sec']/60:.1f} min |"
    )

print("\n### Sample Before/After Reasoning Traces\n")
for idx, trace in enumerate(main_run.get("qualitative_traces", [])[:2], start=1):
    print(f"--- Example {idx}: Numbers={trace['numbers']}, Target={trace['target']} (Ref: {trace['reference_solution']}) ---")
    print(f"[BEFORE | correct={trace['before']['is_correct']} | reward={trace['before']['reward']:.2f}]")
    print(trace["before"]["completion"])
    print(f"\n[AFTER | correct={trace['after']['is_correct']} | reward={trace['after']['reward']:.2f}]")
    print(trace["after"]["completion"])
    print()

zip_path = Path("t4_grpo_artifacts.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in Path("assets").glob("*.png"):
        zf.write(p, arcname=str(p))
    res_file = Path("experiments/results.json")
    if res_file.exists():
        zf.write(res_file, arcname=str(res_file))
print(f"Packaged artifacts into {zip_path} ({zip_path.stat().st_size / 1024:.1f} KB)")

try:
    from google.colab import files
    files.download(str(zip_path))
except ImportError:
    print("Not running inside Google Colab; skipping automatic browser download.")